In [1]:
import os
import numpy as np

import qkeras
from qkeras import QActivation
from qkeras import QConv1D
from qkeras.quantizers import quantized_bits, quantized_relu, quantized_tanh

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, UpSampling1D, Lambda
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, CSVLogger



filters = 4
name =f"Simple_{filters}"
X_total = np.load(f"../Data/X_Data_Bank_Final.npy")
Y_total = np.load(f"../Data/Y_Data_Bank_Final.npy")
X_data = X_total[:1500,:,:]
y_data = Y_total[:1500,:,:]
TIME_STEPS = np.shape(X_data)[1]

VAL_SPLIT = 0.1

SAVE_DIR_opt = f'../Extra_Files/Output_drives/1Conv_checkpoints_running'
LOG_FILE_opt = f'../Extra_Files/Log_files/1Conv_3pulse_noise_tb23_{name}.csv'
MODEL_NAME_TEMPLATE_opt = '1Conv_2pulse_noise.loss_{val_loss:01.5f}.e{epoch:03d}_deconv_' + f'{name}.h5'
checkpoint_path_opt = os.path.join(SAVE_DIR_opt, MODEL_NAME_TEMPLATE_opt)


wq = quantized_bits(8, 2, alpha=1) 
aq = quantized_relu(6) 

qmodel = Sequential([
    Input(shape=(TIME_STEPS, 1)),

    QConv1D(filters, 3, padding='same',
            kernel_quantizer=wq, bias_quantizer=wq),
    QActivation(aq),
    
    QConv1D(filters, 3, padding='same',
            kernel_quantizer=wq, bias_quantizer=wq),
    QActivation(aq),

    QConv1D(filters, 3, padding='same',
            kernel_quantizer=wq, bias_quantizer=wq),
    QActivation(aq),
    
    QConv1D(1, 1, padding='same',
            kernel_quantizer=wq, bias_quantizer=wq),
    
    QActivation("quantized_sigmoid")
    ])

qmodel.summary()


optimizer = Adam(learning_rate=0.001)
qmodel.compile(loss='binary_crossentropy', optimizer=optimizer) 


if not os.path.isdir(SAVE_DIR_opt):
    os.makedirs(SAVE_DIR_opt)

callbacks = [
    ModelCheckpoint(checkpoint_path_opt, monitor="val_loss", save_best_only=True, mode="min", verbose=1),
    CSVLogger(LOG_FILE_opt, append=True, separator=','),
    EarlyStopping(monitor="val_loss", mode="min", patience=20, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=20, min_lr=1e-6, verbose=1) 
]


qmodel.fit(
    X_data, y_data,
    epochs=300,                  
    shuffle=True,
    validation_split=VAL_SPLIT,
    callbacks=callbacks
)

qmodel.save(f"../Models/QFast_{filters}.h5")

2026-05-28 14:14:27.442546: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 AVX_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-28 14:14:27.481618: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-28 14:14:28.581461: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 AVX_VNNI FMA
To enable them in other operations, rebuild TensorFlow with 

Instructions for updating:
Lambda fuctions will be no more assumed to be used in the statement where they are used, or at least in the same block. https://github.com/tensorflow/tensorflow/issues/56089
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 q_conv1d (QConv1D)          (None, 80, 4)             16        
                                                                 
 q_activation (QActivation)  (None, 80, 4)             0         
                                                                 
 q_conv1d_1 (QConv1D)        (None, 80, 4)             52        
                                                                 
 q_activation_1 (QActivation  (None, 80, 4)            0         
 )                                                               
                                                                 
 q_conv1d_2 (QConv1D)        (None, 80, 4)           

 1/43 [..............................] - ETA: 0s - loss: 0.2250
Epoch 18: val_loss improved from 0.20420 to 0.20160, saving model to ../Extra_Files/Output_drives/1Conv_checkpoints_running/1Conv_2pulse_noise.loss_0.20160.e018_deconv_Simple_4.h5
43/43 [==============================] - 0s 2ms/step - loss: 0.2031 - val_loss: 0.2016 - lr: 0.0010
Epoch 19/300
 1/43 [..............................] - ETA: 0s - loss: 0.1799
Epoch 19: val_loss improved from 0.20160 to 0.19590, saving model to ../Extra_Files/Output_drives/1Conv_checkpoints_running/1Conv_2pulse_noise.loss_0.19590.e019_deconv_Simple_4.h5
43/43 [==============================] - 0s 2ms/step - loss: 0.1983 - val_loss: 0.1959 - lr: 0.0010
Epoch 20/300
 1/43 [..............................] - ETA: 0s - loss: 0.1783
Epoch 20: val_loss improved from 0.19590 to 0.19321, saving model to ../Extra_Files/Output_drives/1Conv_checkpoints_running/1Conv_2pulse_noise.loss_0.19321.e020_deconv_Simple_4.h5
43/43 [==============================] - 0

43/43 [==============================] - 0s 2ms/step - loss: 0.1638 - val_loss: 0.1640 - lr: 0.0010
Epoch 43/300
 1/43 [..............................] - ETA: 0s - loss: 0.1954
Epoch 43: val_loss did not improve from 0.16403
43/43 [==============================] - 0s 1ms/step - loss: 0.1667 - val_loss: 0.1666 - lr: 0.0010
Epoch 44/300
 1/43 [..............................] - ETA: 0s - loss: 0.1751
Epoch 44: val_loss improved from 0.16403 to 0.16106, saving model to ../Extra_Files/Output_drives/1Conv_checkpoints_running/1Conv_2pulse_noise.loss_0.16106.e044_deconv_Simple_4.h5
43/43 [==============================] - 0s 2ms/step - loss: 0.1647 - val_loss: 0.1611 - lr: 0.0010
Epoch 45/300
 1/43 [..............................] - ETA: 0s - loss: 0.1461
Epoch 45: val_loss improved from 0.16106 to 0.16070, saving model to ../Extra_Files/Output_drives/1Conv_checkpoints_running/1Conv_2pulse_noise.loss_0.16070.e045_deconv_Simple_4.h5
43/43 [==============================] - 0s 2ms/step - loss: 

43/43 [==============================] - 0s 2ms/step - loss: 0.1485 - val_loss: 0.1502 - lr: 0.0010
Epoch 71/300
 1/43 [..............................] - ETA: 0s - loss: 0.1630
Epoch 71: val_loss did not improve from 0.15024
43/43 [==============================] - 0s 2ms/step - loss: 0.1479 - val_loss: 0.1541 - lr: 0.0010
Epoch 72/300
 1/43 [..............................] - ETA: 0s - loss: 0.1539
Epoch 72: val_loss did not improve from 0.15024
43/43 [==============================] - 0s 1ms/step - loss: 0.1478 - val_loss: 0.1512 - lr: 0.0010
Epoch 73/300
 1/43 [..............................] - ETA: 0s - loss: 0.1509
Epoch 73: val_loss improved from 0.15024 to 0.14869, saving model to ../Extra_Files/Output_drives/1Conv_checkpoints_running/1Conv_2pulse_noise.loss_0.14869.e073_deconv_Simple_4.h5
43/43 [==============================] - 0s 2ms/step - loss: 0.1486 - val_loss: 0.1487 - lr: 0.0010
Epoch 74/300
 1/43 [..............................] - ETA: 0s - loss: 0.1627
Epoch 74: val_lo

43/43 [==============================] - 0s 2ms/step - loss: 0.1367 - val_loss: 0.1361 - lr: 0.0010
Epoch 101/300
 1/43 [..............................] - ETA: 0s - loss: 0.1408
Epoch 101: val_loss improved from 0.13609 to 0.13556, saving model to ../Extra_Files/Output_drives/1Conv_checkpoints_running/1Conv_2pulse_noise.loss_0.13556.e101_deconv_Simple_4.h5
43/43 [==============================] - 0s 2ms/step - loss: 0.1370 - val_loss: 0.1356 - lr: 0.0010
Epoch 102/300
 1/43 [..............................] - ETA: 0s - loss: 0.1251
Epoch 102: val_loss did not improve from 0.13556
43/43 [==============================] - 0s 1ms/step - loss: 0.1369 - val_loss: 0.1378 - lr: 0.0010
Epoch 103/300
 1/43 [..............................] - ETA: 0s - loss: 0.1364
Epoch 103: val_loss did not improve from 0.13556
43/43 [==============================] - 0s 2ms/step - loss: 0.1367 - val_loss: 0.1373 - lr: 0.0010
Epoch 104/300
 1/43 [..............................] - ETA: 0s - loss: 0.1105
Epoch 104

43/43 [==============================] - 0s 1ms/step - loss: 0.1322 - val_loss: 0.1325 - lr: 0.0010
Epoch 132/300
 1/43 [..............................] - ETA: 0s - loss: 0.1334
Epoch 132: val_loss did not improve from 0.13194
43/43 [==============================] - 0s 2ms/step - loss: 0.1335 - val_loss: 0.1322 - lr: 0.0010
Epoch 133/300
 1/43 [..............................] - ETA: 0s - loss: 0.1234
Epoch 133: val_loss did not improve from 0.13194
43/43 [==============================] - 0s 1ms/step - loss: 0.1324 - val_loss: 0.1322 - lr: 0.0010
Epoch 134/300
 1/43 [..............................] - ETA: 0s - loss: 0.0961
Epoch 134: val_loss improved from 0.13194 to 0.13146, saving model to ../Extra_Files/Output_drives/1Conv_checkpoints_running/1Conv_2pulse_noise.loss_0.13146.e134_deconv_Simple_4.h5
43/43 [==============================] - 0s 2ms/step - loss: 0.1320 - val_loss: 0.1315 - lr: 0.0010
Epoch 135/300
 1/43 [..............................] - ETA: 0s - loss: 0.1386
Epoch 135

Epoch 165/300
 1/43 [..............................] - ETA: 0s - loss: 0.1293
Epoch 165: val_loss did not improve from 0.12978
43/43 [==============================] - 0s 1ms/step - loss: 0.1312 - val_loss: 0.1319 - lr: 0.0010
Epoch 166/300
 1/43 [..............................] - ETA: 0s - loss: 0.1310
Epoch 166: val_loss did not improve from 0.12978
43/43 [==============================] - 0s 2ms/step - loss: 0.1307 - val_loss: 0.1325 - lr: 0.0010
Epoch 167/300
 1/43 [..............................] - ETA: 0s - loss: 0.1336
Epoch 167: val_loss did not improve from 0.12978
43/43 [==============================] - 0s 1ms/step - loss: 0.1313 - val_loss: 0.1307 - lr: 0.0010
Epoch 168/300
 1/43 [..............................] - ETA: 0s - loss: 0.1343
Epoch 168: val_loss did not improve from 0.12978
43/43 [==============================] - 0s 1ms/step - loss: 0.1306 - val_loss: 0.1329 - lr: 0.0010
Epoch 169/300
 1/43 [..............................] - ETA: 0s - loss: 0.1235
Epoch 169: val

In [2]:
import hls4ml
from qkeras.utils import _add_supported_quantized_objects

co = {}
_add_supported_quantized_objects(co)
print(co)
hls_config = hls4ml.utils.config_from_keras_model(
    qmodel,
    granularity='name',
    backend='Vitis'
)
hls_model = hls4ml.converters.convert_from_keras_model(
    qmodel,
    hls_config=hls_config,
    backend='Vitis',
    output_dir='../HLS_models/model_final',
    part='xcu250-figd2104-2L-e',
    io_type='io_stream',
)
hls_model.compile()

Matplotlib created a temporary config/cache directory at /tmp/matplotlib-o4ity7y2 because the default path (/home/jovyan/.cache/matplotlib) is not a writable directory; it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
/opt/conda/lib/python3.10/site-packages/hls4ml/converters/__init__.py:27: UserWarning: WARNING: Pytorch converter is not enabled!
  warnings.warn("WARNING: Pytorch converter is not enabled!", stacklevel=1)


{'QInitializer': <class 'qkeras.qlayers.QInitializer'>, 'QDense': <class 'qkeras.qlayers.QDense'>, 'QConv1D': <class 'qkeras.qconvolutional.QConv1D'>, 'QConv2D': <class 'qkeras.qconvolutional.QConv2D'>, 'QConv2DTranspose': <class 'qkeras.qconvolutional.QConv2DTranspose'>, 'QSimpleRNNCell': <class 'qkeras.qrecurrent.QSimpleRNNCell'>, 'QSimpleRNN': <class 'qkeras.qrecurrent.QSimpleRNN'>, 'QLSTMCell': <class 'qkeras.qrecurrent.QLSTMCell'>, 'QLSTM': <class 'qkeras.qrecurrent.QLSTM'>, 'QGRUCell': <class 'qkeras.qrecurrent.QGRUCell'>, 'QGRU': <class 'qkeras.qrecurrent.QGRU'>, 'QBidirectional': <class 'qkeras.qrecurrent.QBidirectional'>, 'QDepthwiseConv2D': <class 'qkeras.qconvolutional.QDepthwiseConv2D'>, 'QSeparableConv1D': <class 'qkeras.qconvolutional.QSeparableConv1D'>, 'QSeparableConv2D': <class 'qkeras.qconvolutional.QSeparableConv2D'>, 'QActivation': <class 'qkeras.qlayers.QActivation'>, 'QAdaptiveActivation': <class 'qkeras.qlayers.QAdaptiveActivation'>, 'QBatchNormalization': <class